# SMOKE TEST: tiny data, 1 epoch, 2 folds. Only checks that the code runs.

# 05 · Second run, H2 to H5: what do the labels and the models actually depend on?

**Paper:** Machuve et al. (2022), Frontiers in Artificial Intelligence 5: 733345. https://doi.org/10.3389/frai.2022.733345

Notebook 03 showed that a model trained on farm labels agrees with the laboratory (PCR) only about 72% of the time.
This notebook tests four explanations. Each has a fixed test and a fixed reading of the result.

| | Question | Test | How to read it |
|---|---|---|---|
| **H2** | Does the floor give the label away? | Train and test on images whose centre is painted grey (only a 16-pixel frame of floor is left). | Far above chance means the scene, not the dropping, carries label information (a shortcut). |
| **H3** | Can PCR labels be learned from photos as well as farm labels can? | 5-fold CV on lab photos only, and on an equal-sized farm sample with the same class counts. | If PCR is much lower, the disease is less visible than the farm labels suggest. |
| **H4** | Were lab and farm photos taken differently? | Logistic regression on frozen ImageNet features predicts farm vs lab. | AUC near 0.5 means the sets look alike; near 1 means they differ, so a farm-trained model faces a new kind of photo. |
| **H5** | How far does colour alone go? | A colour histogram plus logistic regression, no deep learning. | The farm labelling rules are colour and shape rules. If colour alone is close to the deep model on farm labels, the deep model is mostly learning those rules. |

All tests use the same clean set and the same 5 folds as notebook 04.

In [ ]:
SMOKE = True         # True = tiny run that only checks the code
SEED = 42
N_FOLDS = 2 if SMOKE else 5
NEAR_DUP_MAX_HAMMING = 4   # same grouping rule as notebook 03

# MobileNetV2 training, the paper's recommended model. Stage 1 trains only the new top; stage 3 unfreezes the
# whole network but keeps the BatchNorm layers frozen (the paper's best setting, and the best in our first run).
LR_HEAD, LR_FINE = 1e-3, 1e-5
EPOCHS_HEAD = 1 if SMOKE else 20
EPOCHS_FINE = 1 if SMOKE else 30
PATIENCE_HEAD, PATIENCE_FINE = 5, 8
BATCH = 64

OUT = "/kaggle/working/results_05"

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import gc, glob, json, time, hashlib, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy import stats
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, recall_score,
                             confusion_matrix, roc_auc_score)

import tensorflow as tf
import keras
from keras import layers

keras.mixed_precision.set_global_policy("mixed_float16")
for g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)
os.makedirs(OUT, exist_ok=True)

versions = {"python": platform.python_version(), "tensorflow": tf.__version__, "keras": keras.__version__,
            "gpus": len(tf.config.list_physical_devices("GPU"))}
print(versions)
json.dump(versions, open(f"{OUT}/versions.json", "w"))

## Data, duplicate groups and folds

This is the same preparation as notebook 03, so the clean set is identical:

1. Read the 8,067 images. A `pcr` prefix in the file name means the label came from the laboratory (PCR).
2. Group images that are the same photo: identical files (same MD5) or nearly identical pictures
   (dHash distance of 4 bits or less out of 64). A group is never split between training and testing.
3. Drop groups whose images carry different labels, and keep one copy of each identical file.
4. Cut the clean set into 5 folds with `StratifiedGroupKFold` (seed 42). Each fold has about the same class mix,
   and every group sits in one fold only. Each fold takes one turn as the test fold.

In [ ]:
csv_path = glob.glob("/kaggle/input/**/train_data.csv", recursive=True)[0]
IMG_DIR = os.path.join(os.path.dirname(csv_path), "Train")
df = pd.read_csv(csv_path)
df.columns = ["file", "label"]
df["source"] = np.where(df["file"].str.startswith("pcr"), "lab", "farm")
CLASSES = sorted(df["label"].unique())
SHORT = ["cocci", "healthy", "ncd", "salmo"]
df["y"] = df["label"].map({c: i for i, c in enumerate(CLASSES)})

if SMOKE:
    df = (df.groupby(["label", "source"], group_keys=False)
            .apply(lambda g: g.sample(min(len(g), 40), random_state=SEED))
            .reset_index(drop=True))

def load(f):
    img = Image.open(os.path.join(IMG_DIR, f)).convert("RGB")
    return np.asarray(img if img.size == (224, 224) else img.resize((224, 224)), dtype=np.uint8)

X = np.stack([load(f) for f in df["file"]])
Y = df["y"].values.astype(np.int32)
df["md5"] = [hashlib.md5(open(os.path.join(IMG_DIR, f), "rb").read()).hexdigest() for f in df["file"]]

def dhash_bits(arr):
    a = np.asarray(Image.fromarray(arr).convert("L").resize((9, 8), Image.LANCZOS), dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).flatten()

B = np.stack([dhash_bits(x) for x in X]).astype(np.float32)
ham = np.rint(B @ (1 - B).T + (1 - B) @ B.T).astype(np.int16)
same = df["md5"].values[:, None] == df["md5"].values[None, :]
link = same | (ham <= NEAR_DUP_MAX_HAMMING)
np.fill_diagonal(link, False)
_, GROUP = connected_components(csr_matrix(link), directed=False)
del ham, same, link

conflict = df.groupby(GROUP)["label"].transform("nunique").values > 1
keep = df.index[~conflict & ~df.duplicated("md5")].values
print(f"{len(df)} images, {conflict.sum()} removed (conflicting labels), {len(keep)} kept")

FOLD = np.full(len(df), -1)
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for k, (_, test_pos) in enumerate(sgkf.split(keep, Y[keep], groups=GROUP[keep])):
    FOLD[keep[test_pos]] = k
df["fold"] = FOLD
df[["file", "label", "source", "md5", "fold"]].assign(group=GROUP).to_csv(f"{OUT}/folds.csv", index=False)
print(pd.crosstab([df.loc[keep, "source"], df.loc[keep, "label"]], df.loc[keep, "fold"]))

## Training recipes and the model

Two recipes are compared. Everything else (model, layers, learning rates, epochs) stays the same.

| | Paper recipe | Corrected recipe |
|---|---|---|
| Pixel scaling | 0 to 1 (as in the paper) | -1 to 1 (what MobileNetV2's ImageNet weights expect) |
| Brightness | multiply by 0.2 to 1.0 (training images are only ever darkened) | multiply by 0.8 to 1.2 (darker or brighter) |
| Flips | vertical | vertical and horizontal |
| Rotation | up to 45 degrees | up to 45 degrees |

The model is MobileNetV2 with the paper's top: global average pooling, dropout 0.2 and a softmax layer.
Training has two steps: the top alone (stage 1), then the whole network with BatchNorm frozen (stage 3).
Validation data (about 1/7 of the training folds, also group-aware) decides when to stop and which epoch to keep.
The test fold is never used for any choice.

`mask_centre=True` paints the middle of every image grey after augmentation, so that only a 16-pixel frame of floor is
left. It is used by the shortcut test in notebook 05.

In [ ]:
RECIPES = {
    "paper":     dict(scale="0_1",  flip="vertical",                rot=45, bright=(0.2, 1.0)),
    "corrected": dict(scale="-1_1", flip="horizontal_and_vertical", rot=45, bright=(0.8, 1.2)),
}
FRAME = 16   # border kept by the centre mask. 32 px was tried in the smoke test: droppings often reached into it
ring = np.zeros((224, 224, 1), np.float32)
ring[:FRAME], ring[-FRAME:], ring[:, :FRAME], ring[:, -FRAME:] = 1, 1, 1, 1
RING = tf.constant(ring)

def make_ds(idx, recipe, train, mask_centre=False, labels=None, batch=BATCH, seed=SEED):
    r = RECIPES[recipe]
    labels = Y[idx] if labels is None else labels
    aug = [layers.RandomFlip(r["flip"], seed=seed, dtype="float32"),
           layers.RandomRotation(r["rot"] / 360, fill_mode="nearest", seed=seed, dtype="float32")] if train else []

    def prep(x, y):
        x = tf.cast(x, tf.float32)
        for a in aug:
            x = tf.cast(a(x, training=True), tf.float32)
        if train:
            f = tf.random.uniform([tf.shape(x)[0], 1, 1, 1], *r["bright"], seed=seed)
            x = tf.clip_by_value(x * f, 0.0, 255.0)
        if mask_centre:
            x = x * RING + 127.5 * (1.0 - RING)       # grey out the centre, keep the frame
        x = x / 255.0 if r["scale"] == "0_1" else x / 127.5 - 1.0
        return x, y

    ds = tf.data.Dataset.from_tensor_slices((X[idx], labels))
    if train:
        ds = ds.shuffle(len(idx), seed=seed, reshuffle_each_iteration=True)
    return ds.batch(batch).map(prep, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

def build_model(n_classes=4):
    base = keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=(224, 224, 3))
    inp = keras.Input((224, 224, 3))
    x = layers.GlobalAveragePooling2D()(base(inp))
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(n_classes, activation="softmax", dtype="float32")(x)
    return keras.Model(inp, out), base

def split_val(pool, seed):
    """Group-aware validation split: about 1/7 of the training pool."""
    cv = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=seed)
    tr, va = next(cv.split(pool, Y[pool], groups=GROUP[pool]))
    return pool[tr], pool[va]

def train(train_idx, val_idx, recipe, seed, mask_centre=False, labels_of=None, n_classes=4, tag=""):
    """Stage 1 (top only) then stage 3 (all layers, BatchNorm frozen). Returns the trained model."""
    keras.utils.set_random_seed(seed)
    model, base = build_model(n_classes)
    lab = (lambda i: Y[i]) if labels_of is None else labels_of
    tr = make_ds(train_idx, recipe, True, mask_centre, lab(train_idx), seed=seed)
    va = make_ds(val_idx, recipe, False, mask_centre, lab(val_idx))
    log = []
    for stage, lr, epochs, patience in ((1, LR_HEAD, EPOCHS_HEAD, PATIENCE_HEAD), (3, LR_FINE, EPOCHS_FINE, PATIENCE_FINE)):
        base.trainable = stage == 3
        if stage == 3:
            for l in base.layers:
                if isinstance(l, layers.BatchNormalization):
                    l.trainable = False
        model.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy",
                      metrics=["accuracy"], jit_compile=False)
        stop = keras.callbacks.EarlyStopping("val_loss", patience=patience, restore_best_weights=True)
        h = model.fit(tr, validation_data=va, epochs=epochs, callbacks=[stop], verbose=0)
        hist = pd.DataFrame(h.history).assign(stage=stage, epoch=lambda d: np.arange(1, len(d) + 1))
        log.append(hist)
        best = int(np.argmin(h.history["val_loss"]))
        print(f"   {tag} stage {stage}: {len(hist)} epochs, best val_loss {h.history['val_loss'][best]:.3f} "
              f"(val acc {h.history['val_accuracy'][best]:.3f})")
    pd.concat(log).to_csv(f"{OUT}/history_{tag}.csv", index=False)
    return model

def predict(model, idx, recipe, mask_centre=False):
    return model.predict(make_ds(idx, recipe, False, mask_centre, np.zeros(len(idx), np.int32)), verbose=0)

def scores(y, p):
    """The metrics we report for every test set."""
    pred = p.argmax(1)
    labels = list(range(p.shape[1]))
    rec = recall_score(y, pred, labels=labels, average=None, zero_division=0)
    out = {"n": len(y), "accuracy": accuracy_score(y, pred), "balanced_accuracy": balanced_accuracy_score(y, pred),
           "macro_f1": f1_score(y, pred, labels=labels, average="macro", zero_division=0)}
    if p.shape[1] == 4:
        out.update({f"recall_{s}": r for s, r in zip(SHORT, rec)})
    return out

def mean_ci(values):
    """Mean and 95% confidence interval over folds (t distribution, n-1 degrees of freedom)."""
    v = np.asarray(values, float)
    if len(v) < 2:
        return v.mean(), np.nan, np.nan
    half = stats.t.ppf(0.975, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v))
    return v.mean(), v.mean() - half, v.mean() + half

## H2 · Shortcut test: floor only

First we look at what the mask leaves. Droppings sometimes reach the edge of the photo, so a few frames still
show part of a dropping. That can only push the floor-only score up, so treat it as an upper limit.

In [ ]:
show = np.random.RandomState(0).choice(keep, 12, replace=False)
fig, ax = plt.subplots(2, 12, figsize=(18, 3.4))   # top: photo, bottom: what the model sees
masked = (X[show].astype(np.float32) * ring + 127.5 * (1 - ring)).astype(np.uint8)
for j, i in enumerate(show):
    ax[0, j].imshow(X[i]); ax[1, j].imshow(masked[j]); ax[0, j].set_title(df.loc[i, "label"][:10], fontsize=7)
    ax[0, j].axis("off"); ax[1, j].axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/mask_examples.png", dpi=120); plt.show()

In [ ]:
rows = []
farm_keep = keep[df.loc[keep, "source"].values == "farm"]
for k in range(N_FOLDS):
    test = farm_keep[FOLD[farm_keep] == k]
    tr, va = split_val(farm_keep[FOLD[farm_keep] != k], SEED + k)
    model = train(tr, va, "corrected", seed=SEED + k, mask_centre=True, tag=f"H2_fold{k}")
    p = predict(model, test, "corrected", mask_centre=True)
    rows.append({"test": "H2_floor_only", "fold": k, **scores(Y[test], p)})
    print(f"   fold {k}: floor-only accuracy {rows[-1]['accuracy']:.4f}, balanced {rows[-1]['balanced_accuracy']:.4f}")
    pd.DataFrame(rows).to_csv(f"{OUT}/scores.csv", index=False)
    del model; keras.backend.clear_session(); gc.collect()

# Chance levels for comparison: always guessing the biggest class, and the balanced-accuracy chance of 25%.
majority = pd.Series(Y[farm_keep]).value_counts(normalize=True).max()
print(f"majority-class accuracy on farm photos: {majority:.4f}; balanced-accuracy chance: 0.25")

## H3 · Are PCR labels learnable from photos?

* **PCR set:** the clean lab photos. Train on 4 folds, test on the fifth.
* **Farm control:** a random sample of clean farm photos with **the same number of photos per class** as the PCR set,
  drawn once (seed 42). Same folds, same training.

The two sets differ only in who gave the label, so a large gap points to the labels, not to the amount of data.

In [ ]:
lab_keep = keep[df.loc[keep, "source"].values == "lab"]
need = pd.Series(Y[lab_keep]).value_counts()
rng = np.random.RandomState(SEED)
farm_match = np.concatenate([rng.choice(farm_keep[Y[farm_keep] == c], min(n, (Y[farm_keep] == c).sum()), replace=False)
                             for c, n in need.items()])
print("PCR set per class:", need.sort_index().to_dict())
print("farm control per class:", pd.Series(Y[farm_match]).value_counts().sort_index().to_dict())

for name, subset in (("H3_pcr_only", lab_keep), ("H3_farm_matched", farm_match)):
    for k in range(N_FOLDS):
        test = subset[FOLD[subset] == k]
        rest = subset[FOLD[subset] != k]
        if len(test) == 0 or len(np.unique(Y[rest])) < 4:
            continue
        tr, va = split_val(rest, SEED + k)
        model = train(tr, va, "corrected", seed=SEED + k, tag=f"{name}_fold{k}")
        p = predict(model, test, "corrected")
        rows.append({"test": name, "fold": k, **scores(Y[test], p)})
        print(f"   {name} fold {k}: accuracy {rows[-1]['accuracy']:.4f}, balanced {rows[-1]['balanced_accuracy']:.4f}")
        pd.DataFrame(rows).to_csv(f"{OUT}/scores.csv", index=False)
        del model; keras.backend.clear_session(); gc.collect()

## H4 and H5 · Feature-based tests (no fine-tuning)

* **Frozen features:** MobileNetV2 with its ImageNet weights, no training. Each photo becomes 1,280 numbers.
* **Colour features:** a 3-D colour histogram in HSV space (8 hue x 4 saturation x 4 value bins = 128 numbers, each
  photo's histogram sums to 1), plus the mean and spread of each RGB channel.
* Both feed a logistic regression (standardised inputs, balanced class weights), scored with the same 5 folds.
  The regression is fitted on the training folds only.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

extractor = keras.applications.MobileNetV2(include_top=False, weights="imagenet", pooling="avg", input_shape=(224, 224, 3))
EMB = extractor.predict(tf.data.Dataset.from_tensor_slices(X[keep]).batch(128)
                        .map(lambda x: tf.cast(x, tf.float32) / 127.5 - 1.0), verbose=0).astype(np.float32)
pos = {i: j for j, i in enumerate(keep)}            # row of each kept image in EMB

def colour_features(arr):
    hsv = np.asarray(Image.fromarray(arr).convert("HSV"), dtype=np.float32) / 256.0
    h, _ = np.histogramdd(hsv.reshape(-1, 3), bins=(8, 4, 4), range=((0, 1), (0, 1), (0, 1)))
    rgb = arr.reshape(-1, 3).astype(np.float32) / 255.0
    return np.concatenate([h.ravel() / h.sum(), rgb.mean(0), rgb.std(0)])

COL = np.stack([colour_features(X[i]) for i in keep]).astype(np.float32)

def cv_logreg(feats, idx, target, name):
    """5-fold logistic regression on precomputed features; returns out-of-fold probabilities."""
    F = feats[[pos[i] for i in idx]]
    oof = np.zeros((len(idx), len(np.unique(target))))
    for k in range(N_FOLDS):
        te, tr = FOLD[idx] == k, FOLD[idx] != k
        if te.sum() == 0:
            continue
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5, class_weight="balanced"))
        clf.fit(F[tr], target[tr])
        oof[te] = clf.predict_proba(F[te])
        rows.append({"test": name, "fold": k, **scores(target[te], oof[te])})
    pd.DataFrame(rows).to_csv(f"{OUT}/scores.csv", index=False)
    return oof

# H4: farm (0) vs lab (1), from frozen deep features and from colour only.
is_lab = (df.loc[keep, "source"].values == "lab").astype(int)
for feats, name in ((EMB, "H4_source_deep"), (COL, "H4_source_colour")):
    p = cv_logreg(feats, keep, is_lab, name)
    auc = roc_auc_score(is_lab, p[:, 1])
    per_class = {CLASSES[c]: round(roc_auc_score(is_lab[Y[keep] == c], p[Y[keep] == c, 1]), 3) for c in range(4)}
    print(f"{name}: farm-vs-lab AUC {auc:.3f}; within each disease class {per_class}")
    json.dump({"auc": auc, "auc_within_class": per_class}, open(f"{OUT}/{name}_auc.json", "w"), indent=1)

# H5: disease label from colour only, and from frozen deep features (a no-fine-tuning reference).
for feats, fname in ((COL, "colour"), (EMB, "deep_frozen")):
    for src, idx in (("farm", farm_keep), ("pcr", lab_keep)):
        cv_logreg(feats, idx, Y[idx], f"H5_{fname}_{src}")

## Summary table

Mean and 95% confidence interval over the folds for every test. Reference numbers for reading them:
chance balanced accuracy is 25%; full-image MobileNetV2 results are in notebook 04.

In [ ]:
res = pd.DataFrame(rows)
out = []
for name, g in res.groupby("test", sort=False):
    row = {"test": name, "folds": len(g), "n_total": int(g["n"].sum())}
    for m in ["accuracy", "balanced_accuracy", "macro_f1"]:
        mu, lo, hi = mean_ci(g[m])
        row[m] = round(100 * mu, 2); row[f"{m}_lo"] = round(100 * lo, 2); row[f"{m}_hi"] = round(100 * hi, 2)
    if "recall_ncd" in g:
        row["recall_ncd"] = round(100 * g["recall_ncd"].mean(), 2)
    out.append(row)
summary = pd.DataFrame(out)
summary.to_csv(f"{OUT}/summary.csv", index=False)
print(summary.to_string(index=False))